# Databricks Unity Catalog

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakelogic/LakeLogic/blob/main/examples/04_cloud_platforms/databricks/unity_catalog/unity_catalog_example.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/04_cloud_platforms/databricks/unity_catalog/unity_catalog_example.ipynb)

## Business Scenario

Databricks Unity Catalog is a common lakehouse control plane. You need a repeatable way to validate and load Delta data without heavyweight Spark jobs.

## Value Proposition

- Contract-driven validation for Unity Catalog tables
- Spark-free Delta operations for smaller workloads
- Consistent lineage and quality checks

---

## Goals

1. Configure Unity Catalog access
2. Validate data with a contract
3. Perform Delta reads and writes


## Setup: Configure Credentials

In [ ]:
import os
from lakelogic import DataProcessor
from lakelogic.engines.delta_adapter import DeltaAdapter
from lakelogic.engines.unity_catalog import resolve_catalog_path
import polars as pl

# Set Databricks credentials
os.environ["DATABRICKS_HOST"] = "https://your-workspace.cloud.databricks.com"
os.environ["DATABRICKS_TOKEN"] = "dapi..."

# Set cloud storage credentials (AWS S3 example)
os.environ["AWS_REGION"] = "us-west-2"
os.environ["AWS_ACCESS_KEY_ID"] = "AKIA..."
os.environ["AWS_SECRET_ACCESS_KEY"] = "..."

print("✅ Credentials configured")

## Example 1: Read Unity Catalog Table

Use Unity Catalog table names directly (`catalog.schema.table`) - LakeLogic automatically resolves them to storage paths!

In [ ]:
# Create processor with Polars engine (no Spark!)
processor = DataProcessor(
    engine="polars",
    contract="unity_catalog_contract.yaml"
)

# Read Unity Catalog table
good_df, bad_df = processor.run_source("main.default.customers")

print(f"✅ Good records: {len(good_df)}")
print(f"❌ Quarantined records: {len(bad_df)}")

# Display good data
good_df.head()

In [ ]:
# Display quarantined data (if any)
if len(bad_df) > 0:
    print("Quarantined records:")
    bad_df.head()
else:
    print("No quarantined records! 🎉")

## Example 2: Table Name Resolution

See how Unity Catalog table names are automatically resolved to storage paths.

In [ ]:
# Resolve Unity Catalog table name
table_name = "main.default.customers"
storage_path = resolve_catalog_path(table_name)

print(f"Table name: {table_name}")
print(f"Storage path: {storage_path}")
print(f"\n✅ LakeLogic automatically handles this resolution!")

## Example 3: MERGE Operation (Upsert)

Perform atomic MERGE operations **without Spark**!

In [ ]:
# Create new/updated customer data
new_customers = pl.DataFrame({
    "customer_id": [1, 2, 999],
    "email": ["alice@example.com", "bob@example.com", "charlie@example.com"],
    "first_name": ["Alice", "Bob", "Charlie"],
    "last_name": ["Smith", "Jones", "Brown"],
    "created_at": ["2026-02-09T10:00:00Z", "2026-02-09T11:00:00Z", "2026-02-09T12:00:00Z"],
    "country": ["US", "UK", "CA"],
    "status": ["active", "active", "active"]
})

print("New/updated customers:")
new_customers

In [ ]:
# MERGE into Unity Catalog table (atomic, no Spark!)
adapter = DeltaAdapter()
stats = adapter.merge(
    target_path="main.default.customers",
    source_df=new_customers,
    merge_key="customer_id"
)

print(f"✅ MERGE complete:")
print(f"  - Updated: {stats['num_updated']} records")
print(f"  - Inserted: {stats['num_inserted']} records")

## Example 4: Time Travel

Access historical versions of your Delta tables.

In [ ]:
# Read specific version
df_v1 = adapter.read("main.default.customers", version=1)
print(f"Version 1: {len(df_v1)} records")
df_v1.head()

In [ ]:
# Read at specific timestamp
df_yesterday = adapter.read(
    "main.default.customers",
    timestamp="2026-02-08T00:00:00Z"
)
print(f"Yesterday: {len(df_yesterday)} records")
df_yesterday.head()

In [ ]:
# Get table history
history = adapter.get_history("main.default.customers", limit=10)
print("Table history (last 10 commits):")
history

## Example 5: Optimize & Vacuum

Maintain your Delta tables for optimal performance.

In [ ]:
# Optimize (compact small files)
print("Optimizing table...")
stats = adapter.optimize("main.default.customers")

print(f"✅ Optimization complete:")
print(f"  - Files added: {stats['num_files_added']}")
print(f"  - Files removed: {stats['num_files_removed']}")

In [ ]:
# Vacuum (delete old files) - dry run first
print("Vacuum dry run (7 days retention)...")
files = adapter.vacuum("main.default.customers", retention_hours=168, dry_run=True)

print(f"Would delete {len(files)} files")
print(f"\nTo actually vacuum, run:")
print(f"adapter.vacuum('main.default.customers', retention_hours=168, dry_run=False)")

## Example 6: Complete Pipeline

Read → Validate → MERGE in one complete workflow.

In [ ]:
# Step 1: Read from Unity Catalog
print("Step 1: Reading from Unity Catalog...")
processor = DataProcessor(
    engine="polars",
    contract="unity_catalog_contract.yaml"
)
good_df, bad_df = processor.run_source("main.default.customers")
print(f"  ✅ Good: {len(good_df)}, ❌ Bad: {len(bad_df)}")

# Step 2: MERGE validated data to silver layer
print("\nStep 2: MERGE validated data to silver layer...")
adapter = DeltaAdapter()
stats = adapter.merge(
    target_path="main.silver.customers",
    source_df=good_df,
    merge_key="customer_id"
)
print(f"  ✅ Updated: {stats['num_updated']}, Inserted: {stats['num_inserted']}")

# Step 3: Write quarantined data
if len(bad_df) > 0:
    print("\nStep 3: Writing quarantined data...")
    adapter.write(
        df=bad_df,
        path="main.quarantine.customers",
        mode="append"
    )
    print(f"  ✅ Wrote {len(bad_df)} quarantined records")

print("\n✅ Pipeline complete!")

## Example 7: Explore Unity Catalog Metadata

Use the Databricks SDK to explore your Unity Catalog.

In [ ]:
from databricks.sdk import WorkspaceClient

# Connect to Databricks
w = WorkspaceClient()

# List catalogs
print("Catalogs:")
for catalog in w.catalogs.list():
    print(f"  - {catalog.name}")

In [ ]:
# List schemas in 'main' catalog
print("Schemas in 'main':")
for schema in w.schemas.list(catalog_name="main"):
    print(f"  - {schema.name}")

In [ ]:
# List tables in 'main.default'
print("Tables in 'main.default':")
for table in w.tables.list(catalog_name="main", schema_name="default"):
    print(f"\n  📊 {table.name}")
    print(f"     Location: {table.storage_location}")
    print(f"     Format: {table.data_source_format}")

## 🎯 Summary

### What We Demonstrated:

✅ **Unity Catalog table names** - Use `catalog.schema.table` directly  
✅ **Spark-free Delta Lake** - Read/write with Polars (10-100x faster)  
✅ **Atomic MERGE** - Upsert operations without Spark  
✅ **Time travel** - Access historical versions  
✅ **Optimize & vacuum** - Maintain table performance  
✅ **Complete pipeline** - Read → Validate → MERGE  

### Learn More:

- 📚 [Delta Lake Support](../../docs/delta_lake_support.md)
- 📚 [Catalog Table Names](../../docs/catalog_table_names.md)
- 📚 [Unity Catalog Contract](unity_catalog_contract.yaml)

---

*Last Updated: February 2026*